# Genotype–Phenotype Heterogeneity in Non-Classical 21-Hydroxylase Deficiency: FAIR^2 Dataset Exploration with `mlcroissant`
This notebook demonstrates loading, reviewing, and exploring the FAIR^2 clinical dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic metadata info
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")
print(f"Date Published: {dataset.metadata.datePublished}")


## 2. Data Overview
Review available record sets, their fields, and IDs for data exploration.

All references to data entities (record sets, fields) are by their `@id` fields.


In [ ]:
# Explore available record sets and their fields
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"  @id: {rs['@id']} | Name: {rs.get('name', rs['@id'])}")
    # List their fields/columns
    print("    Fields/Columns:")
    for field in rs['fields']:
        print(f"      @id: {field['@id']}, Name: {field.get('name', field['@id'])}, Data type: {field.get('dataType')}")
    print("")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis. All references use `@id` fields.

Below we extract data for all available record sets.

In [ ]:
# Prepare a list of record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for '{record_set_id}' ({len(records)} records)")
        print(f"Columns (@id): {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head())
    else:
        print(f"No records found for '{record_set_id}'.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps using field `@id`s.

- Filter numeric fields
- Normalize values
- Group records by categorical field

We'll select a numeric field and a group field for demonstration. Record set and fields are referenced by `@id`. You may adjust these selections to match the actual loaded data.


In [ ]:
# Example EDA for the first loaded record set
if dataframes:
    # Use the first record set
    first_record_set_id = list(dataframes.keys())[0]
    df = dataframes[first_record_set_id]

    # Try to select a numeric field and a group field based on existing columns
    numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
    else:
        numeric_field_id = df.columns[0]  # Fallback

    # Try to pick a categorical/grouping field
    group_fields = [col for col in df.columns if df[col].dtype == 'object']
    if group_fields:
        group_field_id = group_fields[0]
    else:
        group_field_id = df.columns[0]

    print(f"Using numeric field: {numeric_field_id}")
    print(f"Using group field: {group_field_id}")

    # Filtering: Suppose threshold is median
    threshold = df[numeric_field_id].median() if df[numeric_field_id].dtype in ['int64','float64'] else None

    if threshold is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Groupby
        if group_field_id in filtered_df.columns:
            try:
                grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
                print(f"Grouped (mean) by '{group_field_id}':")
                print(grouped_df.head())
            except Exception as e:
                print(f"Groupby error: {e}")
    else:
        print("No numeric field found for filtering.")
else:
    print("No DataFrames available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below is an example using Matplotlib to plot the distribution of the chosen numeric field, referenced by its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Use variables from previous analysis
    df = dataframes[first_record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(6,4))
        sns.histplot(df[numeric_field_id].dropna(), kde=True)
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()
    else:
        print(f"Numeric field '{numeric_field_id}' not found in columns.")
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook showcased how to load, extract, and explore the FAIR^2 dataset using `mlcroissant`, referencing all entities by their `@id` field as required by the Croissant standard.

Key steps included:
- Loading dataset metadata and reviewing contents.
- Enumerating record sets and their fields by `@id`.
- Extracting and visualizing records for basic clinical analysis.
- Performing EDA and grouping analyses based on clinical feature identifiers.

The FAIR^2 dataset demonstrates structured FAIR clinical tabulation, suitable for research, illustration, and reproducibility.
